# Étape 2 — Analyse Exploratoire (EDA)
**Projet :** Génération de données synthétiques de sinistres pour la tarification en assurance  
**Dataset :** Insurance Claims Data — 58 592 polices, 41 variables  
**Auteurs :** Groupe ISFA 2025-2026

---

### Contenu du notebook
1. Chargement des données
2. Statistiques descriptives
3. Distribution de la variable cible
4. Distributions des variables numériques clés
5. Détection des valeurs aberrantes (boxplots)
6. Matrice de corrélations
7. Corrélations avec claim_status
8. Comparaison sinistres vs non-sinistres (test Mann-Whitney)
9. Taux de sinistres par variables catégorielles
10. Sélection de variables (Pearson + Spearman)


## 0. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Configuration graphique
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11})
COLORS = {'sinistre': '#E74C3C', 'non_sinistre': '#2E86AB', 'neutre': '#1F4E79'}

os.makedirs('../outputs/figures', exist_ok=True)
print("✓ Imports et configuration OK")

## 1. Chargement des données
On charge `data_encoded.csv` — le dataset après prétraitement de l'étape 1 (encodage, parsing, normalisation exclue pour l'EDA).

In [ ]:
df = pd.read_csv('../outputs/data_encoded.csv')
print(f"Shape : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

# Séparation par classe
df0 = df[df['claim_status'] == 0]
df1 = df[df['claim_status'] == 1]
print(f"Non-sinistres (0) : {len(df0):,} ({len(df0)/len(df)*100:.1f}%)")
print(f"Sinistres     (1) : {len(df1):,} ({len(df1)/len(df)*100:.1f}%)")

# Colonnes numériques originales
numeric_cols = [
    'subscription_length', 'vehicle_age', 'customer_age', 'region_density',
    'displacement', 'cylinder', 'turning_radius', 'length', 'width',
    'gross_weight', 'torque_nm', 'torque_rpm', 'power_bhp', 'power_rpm',
    'airbags', 'ncap_rating'
]

## 2. Statistiques descriptives
Calcul des statistiques de base sur toutes les variables numériques, enrichies par la skewness (asymétrie) et le kurtosis (aplatissement).

In [ ]:
desc = df[numeric_cols].describe().T
desc['skewness'] = df[numeric_cols].skew()
desc['kurtosis'] = df[numeric_cols].kurtosis()
desc[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']].round(2)

In [ ]:
# Sauvegarde
desc.to_csv('../outputs/statistiques_descriptives.csv')
print("✓ Sauvegardé : outputs/statistiques_descriptives.csv")

# Observations clés
print("\nObservations clés :")
print(f"  subscription_length : skewness={desc.loc['subscription_length','skewness']:.2f} → distribution quasi-uniforme")
print(f"  vehicle_age         : skewness={desc.loc['vehicle_age','skewness']:.2f}, kurtosis={desc.loc['vehicle_age','kurtosis']:.2f} → queue à droite (vieux véhicules)")
print(f"  region_density      : skewness={desc.loc['region_density','skewness']:.2f} → très asymétrique (zones urbaines denses)")

## 3. Distribution de la variable cible — claim_status
Visualisation du déséquilibre entre sinistres et non-sinistres.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Distribution de la variable cible — claim_status",
             fontsize=14, fontweight='bold', color=COLORS['neutre'])

counts = df['claim_status'].value_counts()

# Camembert
axes[0].pie(
    counts, labels=['Non-sinistre (0)', 'Sinistre (1)'],
    colors=[COLORS['non_sinistre'], COLORS['sinistre']],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title("Répartition globale")

# Barres
bars = axes[1].bar(['Non-sinistre (0)', 'Sinistre (1)'], counts.values,
                    color=[COLORS['non_sinistre'], COLORS['sinistre']],
                    edgecolor='white', linewidth=1.5)
axes[1].set_title("Effectifs par classe")
axes[1].set_ylabel("Nombre de polices")
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}', ha='center', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../outputs/figures/01_distribution_cible.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nDéséquilibre : 1 sinistre pour {int(counts[0]/counts[1])} non-sinistres")

## 4. Distributions des variables numériques clés
Comparaison des distributions entre sinistres (rouge) et non-sinistres (bleu) pour 8 variables principales.

In [ ]:
key_vars = ['customer_age', 'vehicle_age', 'subscription_length',
            'region_density', 'power_bhp', 'torque_nm',
            'displacement', 'gross_weight']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Distributions des variables numériques clés",
             fontsize=14, fontweight='bold', color=COLORS['neutre'])
axes = axes.flatten()

for i, col in enumerate(key_vars):
    axes[i].hist(df1[col], bins=40, alpha=0.6, color=COLORS['sinistre'],
                 label='Sinistre', density=True)
    axes[i].hist(df0[col], bins=40, alpha=0.4, color=COLORS['non_sinistre'],
                 label='Non-sinistre', density=True)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel("Densité")
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/figures/02_distributions_numeriques.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Détection des valeurs aberrantes — Boxplots
Identification des outliers par classe via la méthode IQR (Inter-Quartile Range).  
Un point est considéré aberrant si : `valeur < Q1 - 1.5×IQR` ou `valeur > Q3 + 1.5×IQR`

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Détection des valeurs aberrantes — Boxplots par classe",
             fontsize=14, fontweight='bold', color=COLORS['neutre'])
axes = axes.flatten()

print("Nombre d'outliers par variable (méthode IQR) :")
for i, col in enumerate(key_vars):
    data_plot = [df0[col].dropna(), df1[col].dropna()]
    bp = axes[i].boxplot(data_plot, patch_artist=True,
                          labels=['Non-sinistre', 'Sinistre'],
                          medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor(COLORS['non_sinistre']); bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor(COLORS['sinistre']);     bp['boxes'][1].set_alpha(0.6)
    axes[i].set_title(col, fontsize=10, fontweight='bold')

    for j, data in enumerate(data_plot):
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        n_out = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
        axes[i].text(j+1, data.max()*0.98, f'{n_out} out.', ha='center', fontsize=7, color='gray')
    
    # Résumé global
    all_data = df[col]
    Q1, Q3 = all_data.quantile(0.25), all_data.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((all_data < Q1 - 1.5*IQR) | (all_data > Q3 + 1.5*IQR)).sum()
    print(f"  {col:<25} : {n_out:>4} outliers ({n_out/len(df)*100:.1f}%)")

plt.tight_layout()
plt.savefig('../outputs/figures/03_boxplots_outliers.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n→ Tous les outliers sont conservés (cas réels pertinents actuariellement)")

## 6. Matrice de corrélations
Heatmap triangulaire des corrélations entre toutes les variables numériques et claim_status.

In [ ]:
corr_data   = df[numeric_cols + ['claim_status']]
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 8}, linewidths=0.5)
ax.set_title("Matrice de corrélations — variables numériques + claim_status",
             fontsize=13, fontweight='bold', color=COLORS['neutre'], pad=15)
plt.tight_layout()
plt.savefig('../outputs/figures/04_matrice_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

# Multi-colinéarité
print("Multi-colinéarité identifiée :")
print("  displacement / torque_nm / power_bhp  → caractéristiques moteur liées")
print("  length / width / gross_weight          → dimensions physiques du véhicule")

## 7. Corrélations avec claim_status
Classement des variables numériques par corrélation absolue avec la variable cible.

In [ ]:
corr_target = corr_matrix['claim_status'].drop('claim_status').abs().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = [COLORS['sinistre'] if v > 0.05 else COLORS['non_sinistre'] for v in corr_target]
bars = ax.barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='white')
ax.axvline(x=0.05, color='gray', linestyle='--', linewidth=1, label='Seuil 0.05')
ax.set_xlabel("Corrélation absolue avec claim_status")
ax.set_title("Corrélation des variables numériques avec claim_status",
             fontsize=13, fontweight='bold', color=COLORS['neutre'])
ax.legend()
for bar, val in zip(bars, corr_target.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/figures/05_correlations_target.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 variables les plus corrélées à claim_status :")
for var, val in corr_target.sort_values(ascending=False).head(5).items():
    print(f"  {var:<25} : r = {val:.4f}")

## 8. Comparaison sinistres vs non-sinistres
Test de Mann-Whitney (non-paramétrique) pour comparer les distributions entre les deux classes.  
- `***` : p < 0.001 — très significatif  
- `**`  : p < 0.01  
- `*`   : p < 0.05  
- `ns`  : non significatif

In [ ]:
compare_vars = ['customer_age', 'vehicle_age', 'subscription_length', 'region_density']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Comparaison des distributions — Sinistres vs Non-sinistres",
             fontsize=13, fontweight='bold', color=COLORS['neutre'])

print("Résultats des tests de Mann-Whitney :")
for i, col in enumerate(compare_vars):
    axes[i].hist(df0[col], bins=30, alpha=0.5, color=COLORS['non_sinistre'],
                 label=f'Non-sinistre', density=True)
    axes[i].hist(df1[col], bins=30, alpha=0.7, color=COLORS['sinistre'],
                 label=f'Sinistre', density=True)
    stat, pval = stats.mannwhitneyu(df0[col], df1[col], alternative='two-sided')
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    axes[i].set_title(f"{col}\np={pval:.2e} {sig}", fontsize=9, fontweight='bold')
    axes[i].set_ylabel("Densité")
    axes[i].legend(fontsize=7)
    print(f"  {col:<25} : p={pval:.2e} {sig}")

plt.tight_layout()
plt.savefig('../outputs/figures/06_comparaison_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Taux de sinistres par variables catégorielles
Comparaison du taux de sinistres par modalité pour 5 variables catégorielles clés.  
La ligne pointillée représente le taux moyen global (6.4%).

In [ ]:
df_raw = pd.read_csv('../data/Insurance claims data.csv', sep=',')
cat_vars = ['fuel_type', 'segment', 'transmission_type', 'rear_brakes_type', 'steering_type']

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
fig.suptitle("Taux de sinistres par variable catégorielle",
             fontsize=13, fontweight='bold', color=COLORS['neutre'])

taux_global = df_raw['claim_status'].mean()
print(f"Taux de sinistres global : {taux_global*100:.1f}%\n")

for i, col in enumerate(cat_vars):
    taux = df_raw.groupby(col)['claim_status'].mean().sort_values(ascending=False)
    bars = axes[i].bar(taux.index, taux.values * 100,
                        color=COLORS['sinistre'], alpha=0.7, edgecolor='white')
    axes[i].axhline(y=taux_global * 100, color='gray', linestyle='--',
                     linewidth=1.5, label=f'Moy. {taux_global*100:.1f}%')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel("Taux de sinistres (%)")
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=7)
    for bar, val in zip(bars, taux.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val*100:.1f}%', ha='center', fontsize=8, fontweight='bold')
    print(f"{col} :")
    for mod, t in taux.items():
        print(f"  {str(mod):<20} : {t*100:.2f}%")
    print()

plt.tight_layout()
plt.savefig('../outputs/figures/07_taux_sinistres_categorielles.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Sélection de variables
Identification des variables les plus informatives via deux méthodes complémentaires :
- **Corrélation de Pearson** : relations linéaires
- **Corrélation de Spearman** : relations monotones (robuste aux outliers et distributions non-normales)

In [ ]:
# Pearson
corr_pearson = df[numeric_cols].corrwith(df['claim_status']).abs().sort_values(ascending=False)

# Spearman
spearman_corr = {}
for col in numeric_cols:
    rho, pval = stats.spearmanr(df[col], df['claim_status'])
    spearman_corr[col] = {'rho': abs(rho), 'pval': pval}
df_spearman = pd.DataFrame(spearman_corr).T.sort_values('rho', ascending=False)

# Tableau synthèse
df_selection = pd.DataFrame({
    'pearson_abs': corr_pearson,
    'spearman_rho': df_spearman['rho'],
    'spearman_pval': df_spearman['pval']
}).sort_values('pearson_abs', ascending=False)
df_selection['significatif'] = df_selection['spearman_pval'] < 0.05
df_selection.to_csv('../outputs/selection_variables.csv')

print("Tableau de sélection des variables :")
print(df_selection.round(4).to_string())
print("\n✓ Sauvegardé : outputs/selection_variables.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Sélection de variables — Corrélations avec claim_status",
             fontsize=13, fontweight='bold', color=COLORS['neutre'])

# Pearson
corr_plot = corr_pearson.sort_values(ascending=True)
colors_p = [COLORS['sinistre'] if v > 0.02 else '#CCCCCC' for v in corr_plot]
axes[0].barh(corr_plot.index, corr_plot.values, color=colors_p, edgecolor='white')
axes[0].axvline(x=0.02, color='gray', linestyle='--', linewidth=1.5, label='Seuil 0.02')
axes[0].set_title("Corrélation de Pearson (valeur absolue)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Corrélation absolue")
axes[0].legend()

# Spearman
spear_plot = df_spearman['rho'].sort_values(ascending=True)
colors_s = [COLORS['sinistre'] if v > 0.02 else '#CCCCCC' for v in spear_plot]
axes[1].barh(spear_plot.index, spear_plot.values, color=colors_s, edgecolor='white')
axes[1].axvline(x=0.02, color='gray', linestyle='--', linewidth=1.5, label='Seuil 0.02')
axes[1].set_title("Corrélation de Spearman (valeur absolue)", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Corrélation absolue")
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/08_selection_variables.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVariables sélectionnées (Pearson > 0.02 ET Spearman significatif) :")
selected = df_selection[(df_selection['pearson_abs'] > 0.02) & (df_selection['significatif'])]
for var, row in selected.iterrows():
    print(f"  {var:<25} : Pearson={row['pearson_abs']:.4f}, Spearman={row['spearman_rho']:.4f}")

## Résumé — Étape 2

In [ ]:
print("=" * 60)
print("  RÉSUMÉ ÉTAPE 2 — ANALYSE EXPLORATOIRE")
print("=" * 60)
print(f"  Dataset          : {df.shape[0]:,} polices × {df.shape[1]} colonnes")
print(f"  Taux de sinistres: {df['claim_status'].mean()*100:.1f}%")
print(f"  Figures générées : 8 (dans outputs/figures/)")
print(f"  Fichiers CSV     : statistiques_descriptives.csv, selection_variables.csv")
print()
print("  Variables les plus discriminantes :")
for var in corr_pearson.head(5).index:
    print(f"    → {var:<25} (r={corr_pearson[var]:.4f})")
print()
print("  Implications pour la modélisation :")
print("    → Faibles corrélations (<0.10) → deep learning justifié")
print("    → Déséquilibre sévère (1/14)   → génération synthétique nécessaire")
print("    → Multi-colinéarité moteur     → absorbée par CTGAN/TVAE")
print()
print("✓ Étape 2 terminée.")